# time-stage-instrumentation — ex1: accumulate per-stage seconds with time.perf_counter

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `time-stage-instrumentation`. Running the final beacon cell reports progress against the `Logging: time-stage instrumentation` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Logging: time-stage instrumentation` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`time-stage-instrumentation`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "time-stage-instrumentation"
DD_SUBTOPIC = "Logging: time-stage instrumentation"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## `time.perf_counter()` stage instrumentation — quick refresher

Profiling a training loop starts with `time.perf_counter()` around named stages: data-load, forward, backward, optimizer-step, log. Sum the elapsed seconds per stage across the run and you get the wall-clock breakdown — usually data-load and forward dominate.

**The recipe:**

```python
stages = {'forward': 0.0, 'backward': 0.0, 'step': 0.0}
for batch in loader:
    t0 = time.perf_counter()
    out = model(batch)
    stages['forward'] += time.perf_counter() - t0

    t0 = time.perf_counter()
    loss.backward()
    stages['backward'] += time.perf_counter() - t0
    ...
```

**Why `perf_counter` not `time.time`.** `perf_counter` is monotonic and has the highest available resolution on each platform; `time.time` can go backwards when the system clock adjusts and is coarse-grained on Windows. ALWAYS use `perf_counter` for elapsed-time measurement.

**Sum elapsed inside the loop**, don't store individual samples — a 100k-step training run would balloon to gigabytes if you stored every per-stage delta. Sum (and optionally count) is enough to compute the mean later.

**Subtraction handles overflow gracefully.** `perf_counter` is monotonic so `t1 - t0` is always >= 0. No clock-skew edge cases.

### Exercise 1 — accumulate per-stage seconds with time.perf_counter

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply `time.perf_counter()` deltas to accumulate elapsed seconds per named stage across a loop, returning the per-stage totals with measured order-of-magnitude correct.
> Keywords: time, perf_counter, profile, stages
> ```

**KCs targeted:** `perf-counter-elapsed`, `per-stage-sum-accumulator`

Implement `ex1_time_three_stages(n_iters, sleep_forward, sleep_backward, sleep_step)`. The canonical per-stage profiling recipe:

1. Initialize `stages = {'forward': 0.0, 'backward': 0.0, 'step': 0.0}`.
2. Loop `for _ in range(n_iters)`:
   - Time a `time.sleep(sleep_forward)` block; ADD the elapsed to `stages['forward']`.
   - Time a `time.sleep(sleep_backward)` block; add to `stages['backward']`.
   - Time a `time.sleep(sleep_step)` block; add to `stages['step']`.
3. Use `time.perf_counter()` for both endpoints of each timed block. NOT `time.time()`.
4. Return `stages`.

The test asserts (a) each stage's total is at least `n_iters * sleep_X` (sleep + measurement overhead means actual is always >= nominal), (b) the totals are within a generous upper bound, and (c) the relative ORDER of magnitudes matches the input sleeps.

In [ ]:
import time

def ex1_time_three_stages(n_iters, sleep_forward, sleep_backward, sleep_step):
    stages = {'forward': 0.0, 'backward': 0.0, 'step': 0.0}
    for _ in range(n_iters):
        t0 = time.perf_counter()
        time.sleep(sleep_forward)
        stages['forward'] += time.perf_counter() - t0

        t0 = time.perf_counter()
        time.sleep(sleep_backward)
        stages['backward'] += time.perf_counter() - t0

        t0 = time.perf_counter()
        time.sleep(sleep_step)
        stages['step'] += time.perf_counter() - t0
    return stages


<details><summary>Solution</summary>

```python
import time

def ex1_time_three_stages(n_iters, sleep_forward, sleep_backward, sleep_step):
    stages = {'forward': 0.0, 'backward': 0.0, 'step': 0.0}
    for _ in range(n_iters):
        t0 = time.perf_counter()
        time.sleep(sleep_forward)
        stages['forward'] += time.perf_counter() - t0

        t0 = time.perf_counter()
        time.sleep(sleep_backward)
        stages['backward'] += time.perf_counter() - t0

        t0 = time.perf_counter()
        time.sleep(sleep_step)
        stages['step'] += time.perf_counter() - t0
    return stages
```

**`perf_counter` for elapsed, `time.time` for wall clock.** `perf_counter` is monotonic and high-resolution; `time.time` can jump (NTP adjustment, daylight savings) and is coarse-grained on Windows. ALWAYS use `perf_counter` for elapsed-time profiling.

**Why ACCUMULATE not store every delta.** A 100k-step training run with 5 stages would store 500k floats — wasteful when you only need the totals. Summing in place keeps memory O(num_stages) regardless of training length.

**The 50x upper bound is conservative.** On a clean CI runner `time.sleep(0.005)` returns in ~5ms + tiny scheduler overhead, well under 2x nominal. The 50x cap absorbs container-level noise and OS jitter without false-failing the test.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()